# Week 42, Day 6: Forge: Week 42 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w042/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
print("1.", 0.005 * 20 - 0.006 * 20, "(negative: net depression)")
print("2.", 0.5 * (0.5 - 1), "(negative: depression)")
print("3.", round((0.6 * 1.5) / (0.2 * 1.5), 3), "(unchanged)")

### Exercise: Part A: principal components by Hebbian learning

Complete the two loops: Oja updates w by eta y (x - y w); Sanger updates W by eta (outer(y, x) - tril(outer(y, y)) W).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(42)
C = np.array([[3, 1.5, 0.5], [1.5, 2, 0.3], [0.5, 0.3, 1]])
X = (np.linalg.cholesky(C) @ r.normal(size=(3, 20000))).T
evals, V = np.linalg.eigh(C)          # ascending order: V[:, -1] is the first PC

w_oja = 0.1 * r.normal(size=3)
for x in X:
    y = w_oja @ x
    ...

W_gha = 0.1 * r.normal(size=(2, 3))
for x in X:
    y = W_gha @ x
    ...

print("Oja: |cos| with PC1 =", round(abs(w_oja @ V[:, -1]) / np.linalg.norm(w_oja), 4), " |w| =", round(np.linalg.norm(w_oja), 3))
print("Sanger: |overlaps| of the two outputs with PC1 and PC2:\n", np.round(np.abs(W_gha @ V[:, ::-1][:, :2]), 3))

In [ ]:
import numpy as np
def test_oja():
    "Oja's rule finds the first principal component with unit length"
    assert abs(w_oja @ V[:, -1]) / np.linalg.norm(w_oja) > 0.99 and abs(np.linalg.norm(w_oja) - 1) < 0.05

def test_sanger():
    "Sanger's rule finds the first two principal components in order"
    M = np.abs(W_gha @ V[:, ::-1][:, :2])
    assert M[0, 0] > 0.97 and M[1, 1] > 0.97 and M[0, 1] < 0.1 and M[1, 0] < 0.1

test_oja()
test_sanger()
print("All checks passed")

### Exercise: Part B: closing one eye

Complete run(): build the eight inputs as [left, right], where left is the pattern during normal rearing and 0.1 times uniform noise during deprivation; then apply the BCM weight and threshold updates.

In [ ]:
x1 = np.array([1.0, 0.8, 0.2, 0.1]); x2 = x1[::-1].copy()

def run(w, theta, steps, md=False, eta=0.005, tau=50):
    for t in range(steps):
        p = x1 if r.random() < 0.5 else x2
        left = ...
        x = np.r_[left, p]
        y = max(0, w @ x)
        ...
    return w, theta

w_norm, th = run(np.full(8, 0.15), 0.5, 20000)
w_md, _ = run(w_norm, th, 20000, md=True)
print("normal rearing:", np.round(w_norm, 2), " left", round(w_norm[:4].sum(), 2), "right", round(w_norm[4:].sum(), 2))
print("after deprivation:", np.round(w_md, 2), " left", round(w_md[:4].sum(), 2), "right", round(w_md[4:].sum(), 2))

In [ ]:
import numpy as np
def test_binocular():
    "after normal rearing the two eyes have equal weights"
    assert abs(w_norm[:4].sum() - w_norm[4:].sum()) < 0.05

def test_md():
    "after deprivation the closed eye's total weight has fallen by more than half and the open eye's has risen"
    assert w_md[:4].sum() < 0.5 * w_norm[:4].sum() and w_md[4:].sum() > w_norm[4:].sum()

test_binocular()
test_md()
print("All checks passed")